File: chapter_06_B.ipynb<br>
Desc: Imblanace Techniques

---

Part A: Example of Over Sampling Method

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import cross_validate
from sklearn.metrics import confusion_matrix
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.metrics import classification_report

from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

from imblearn.over_sampling import RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import SMOTE

In [ ]:
##--- Prepare cross validation data ---##
df_cross = pd.read_csv('dataset_cross.csv')

#--- Define data type ---#
df_cross = df_cross.astype({"SEX": "category"})
df_cross = df_cross.astype({"NO_OF_DEPENDENT": np.int64})
df_cross = df_cross.astype({"MARITAL_STATUS": "category"})
df_cross = df_cross.astype({"YRS_IN_PRESENT_JOB": np.int64})
df_cross = df_cross.astype({"AGE": np.int64})
df_cross = df_cross.astype({"EDUCATION": "category"})
df_cross = df_cross.astype({"YEARS_OF_EXPERIENCE": np.int64})
df_cross = df_cross.astype({"TOTAL_MONTHLY_INCOME": np.int64})
df_cross = df_cross.astype({"LOAN_AMOUNT": np.int64})
df_cross = df_cross.astype({"COMPANY_TYPE": "category"})
df_cross = df_cross.astype({"LABEL": np.int64})

#--- Encode one-hot teatures ---#
df_cross = pd.get_dummies(df_cross, dtype=int)

#--- Prepare cross validation data ---#
X_cross = df_cross.drop(['LABEL'], axis=1)
y_cross = df_cross['LABEL']

#--- Resample cross validation data ---#
ros = RandomOverSampler(random_state=0)
X_cross_resampled, y_cross_resampled = ros.fit_resample(X_cross, y_cross)

In [ ]:
# Investigate original vs resample class size
print("Original class size")
print(y_cross.value_counts())
print("-------------------")
print("Resampled class size")
print(y_cross_resampled.value_counts())

In [ ]:
##--- Prepare final validation data ---#
df_final = pd.read_csv('dataset_final.csv')

#--- Define data type ---#
df_final = df_final.astype({"SEX": "category"})
df_final = df_final.astype({"NO_OF_DEPENDENT": np.int64})
df_final = df_final.astype({"MARITAL_STATUS": "category"})
df_final = df_final.astype({"YRS_IN_PRESENT_JOB": np.int64})
df_final = df_final.astype({"AGE": np.int64})
df_final = df_final.astype({"EDUCATION": "category"})
df_final = df_final.astype({"YEARS_OF_EXPERIENCE": np.int64})
df_final = df_final.astype({"TOTAL_MONTHLY_INCOME": np.int64})
df_final = df_final.astype({"LOAN_AMOUNT": np.int64})
df_final = df_final.astype({"COMPANY_TYPE": "category"})
df_final = df_final.astype({"LABEL": np.int64})

#--- Encode one-hot features ---#
df_final = pd.get_dummies(df_final, dtype=int)

#--- Prepare cross validation data ---#
X_final = df_final.drop(['LABEL'], axis=1)
y_final = df_final['LABEL']

A1 : Random Forest

In [ ]:
##--- Random Forest cross validation process ---#
f1_list = []
accuracy_list = []
precision_list = []
recall_list = []
max_estimators_list = []

for i in range(1, 10):
    clf = RandomForestClassifier(n_estimators=i, criterion='gini')
    scoring = ['f1', 'accuracy', 'precision', 'recall']
    scores = cross_validate(clf, X_cross_resampled, y_cross_resampled, cv=5, scoring=scoring)
    f1_list.append(np.mean(scores['test_f1']))
    accuracy_list.append(np.mean(scores['test_accuracy']))
    precision_list.append(np.mean(scores['test_precision']))
    recall_list.append(np.mean(scores['test_recall']))
    max_estimators_list.append(i)

fig = plt.figure(figsize = (35,5))
plt.subplot(141)
plt.plot(max_estimators_list, f1_list) 
plt.grid() 
plt.title('F1')
plt.xlabel('n_estimator')
plt.ylabel('F1')

plt.subplot(142)
plt.plot(max_estimators_list, accuracy_list) 
plt.grid() 
plt.title('Acuracy')
plt.xlabel('n_estimator')
plt.ylabel('Accuracy')

plt.subplot(143) 
plt.plot(max_estimators_list, precision_list)
plt.grid()
plt.title('Precision')
plt.xlabel('n_estimator')
plt.ylabel('Precision')

plt.subplot(144) 
plt.plot(max_estimators_list, recall_list)
plt.grid()
plt.title('Recall')
plt.xlabel('n_estimator')
plt.ylabel('Recall')

In [ ]:
##--- Final validation process ---#
clf = RandomForestClassifier(n_estimators=20, criterion='gini', max_depth=40)
clf.fit(X_cross_resampled, y_cross_resampled)
y_pred = clf.predict(X_final)

target_names = ['class 0', 'class 1']
print(classification_report(y_final, y_pred, target_names=target_names, digits=5))

cm = confusion_matrix(y_final, y_pred)
cm_display = ConfusionMatrixDisplay(cm).plot()

A2 : XGBoost

In [ ]:
##--- XGBoost cross validation process ---#
f1_list = []
accuracy_list = []
precision_list = []
recall_list = []
max_estimators_list = []

for i in range(1, 10):
    clf = XGBClassifier(n_estimators=40, max_depth=i, objective='binary:logistic')
    scoring = ['f1', 'accuracy', 'precision', 'recall']
    scores = cross_validate(clf, X_cross_resampled, y_cross_resampled, cv=5, scoring=scoring)
    f1_list.append(np.mean(scores['test_f1']))
    accuracy_list.append(np.mean(scores['test_accuracy']))
    precision_list.append(np.mean(scores['test_precision']))
    recall_list.append(np.mean(scores['test_recall']))
    max_estimators_list.append(i)

fig = plt.figure(figsize = (35,5))
plt.subplot(141)
plt.plot(max_estimators_list, f1_list) 
plt.grid() 
plt.title('F1')
plt.xlabel('n_estimator')
plt.ylabel('F1')

plt.subplot(142)
plt.plot(max_estimators_list, accuracy_list) 
plt.grid() 
plt.title('Acuracy')
plt.xlabel('n_estimator')
plt.ylabel('Accuracy')

plt.subplot(143) 
plt.plot(max_estimators_list, precision_list)
plt.grid()
plt.title('Precision')
plt.xlabel('n_estimator')
plt.ylabel('Precision')

plt.subplot(144) 
plt.plot(max_estimators_list, recall_list)
plt.grid()
plt.title('Recall')
plt.xlabel('n_estimator')
plt.ylabel('Recall')

In [ ]:
##--- XGBoost final validation process ---#
clf = XGBClassifier(n_estimators=40, max_depth=30, objective='binary:logistic')
clf.fit(X_cross_resampled, y_cross_resampled)
y_pred = clf.predict(X_final)

target_names = ['class 0', 'class 1']
print(classification_report(y_final, y_pred, target_names=target_names, digits=5))

cm = confusion_matrix(y_final, y_pred)
cm_display = ConfusionMatrixDisplay(cm).plot()


---

Part B : Example of Under Sampling Method

In [ ]:
##--- Prepare cross validation data ---##
df_cross = pd.read_csv('dataset_cross.csv')

#--- Define data type ---#
df_cross = df_cross.astype({"SEX": "category"})
df_cross = df_cross.astype({"NO_OF_DEPENDENT": np.int64})
df_cross = df_cross.astype({"MARITAL_STATUS": "category"})
df_cross = df_cross.astype({"YRS_IN_PRESENT_JOB": np.int64})
df_cross = df_cross.astype({"AGE": np.int64})
df_cross = df_cross.astype({"EDUCATION": "category"})
df_cross = df_cross.astype({"YEARS_OF_EXPERIENCE": np.int64})
df_cross = df_cross.astype({"TOTAL_MONTHLY_INCOME": np.int64})
df_cross = df_cross.astype({"LOAN_AMOUNT": np.int64})
df_cross = df_cross.astype({"COMPANY_TYPE": "category"})
df_cross = df_cross.astype({"LABEL": np.int64})

#--- Encode one-hot teatures ---#
df_cross = pd.get_dummies(df_cross, dtype=int)

#--- Prepare cross validation data ---#
X_cross = df_cross.drop(['LABEL'], axis=1)
y_cross = df_cross['LABEL']

#--- Resample cross validation data ---#
rus = RandomUnderSampler(random_state=0, replacement=True)
X_cross_resampled, y_cross_resampled = rus.fit_resample(X_cross, y_cross)

In [ ]:
# Investigate original vs resample class size
print("Original class size")
print(y_cross.value_counts())
print("-------------------")
print("Resampled class size")
print(y_cross_resampled.value_counts())

---

Part C : Example of SMOTE method

In [ ]:
##--- Prepare cross validation data ---##
df_cross = pd.read_csv('dataset_cross.csv')

#--- Define data type ---#
df_cross = df_cross.astype({"SEX": "category"})
df_cross = df_cross.astype({"NO_OF_DEPENDENT": np.int64})
df_cross = df_cross.astype({"MARITAL_STATUS": "category"})
df_cross = df_cross.astype({"YRS_IN_PRESENT_JOB": np.int64})
df_cross = df_cross.astype({"AGE": np.int64})
df_cross = df_cross.astype({"EDUCATION": "category"})
df_cross = df_cross.astype({"YEARS_OF_EXPERIENCE": np.int64})
df_cross = df_cross.astype({"TOTAL_MONTHLY_INCOME": np.int64})
df_cross = df_cross.astype({"LOAN_AMOUNT": np.int64})
df_cross = df_cross.astype({"COMPANY_TYPE": "category"})
df_cross = df_cross.astype({"LABEL": np.int64})

#--- Encode one-hot teatures ---#
df_cross = pd.get_dummies(df_cross, dtype=int)

#--- Prepare cross validation data ---#
X_cross = df_cross.drop(['LABEL'], axis=1)
y_cross = df_cross['LABEL']

#--- Resample cross validation data ---#
sm = SMOTE(random_state=42)
X_cross_resampled, y_cross_resampled = sm.fit_resample(X_cross, y_cross)

In [ ]:
# Investigate original vs resample class size
print("Original class size")
print(y_cross.value_counts())
print("-------------------")
print("Resampled class size")
print(y_cross_resampled.value_counts())

---